# Setting config


In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

# import lib python
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine

# immport utils ca nhan
from utils.config import *
from utils.db import *

# import log
import logging
import os

# Log

In [2]:
# Log setting

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%d-%m_%Y %H:%M:%S',
    handlers=[
        logging.FileHandler(
            f'log/ETL_Project_{datetime.now():%Y%m%d}.log'
        ),
        logging.StreamHandler()
    ]
)

log = logging.getLogger(__name__)

# Extract

In [3]:
def extract():
    log.info("Extract started!")

    df_d_center = pd.read_csv('data/raw/distribution_centers.csv')
    log.info(f'Get data distribution_centers done: Total {len(df_d_center)} rows')

    df_i_items = pd.read_csv('data/raw/inventory_items.csv')
    log.info(f'Get data inventory_items done: Total {len(df_i_items)} rows')

    df_o_items = pd.read_csv('data/raw/order_items.csv')
    log.info(f'Get data order_items done: Total {len(df_o_items)} rows')

    df_orders = pd.read_csv('data/raw/orders.csv')
    log.info(f'Get data orders done: Total {len(df_orders)} rows')

    df_products = pd.read_csv('data/raw/products.csv')
    log.info(f'Get data products done: Total {len(df_products)} rows')

    df_users = pd.read_csv('data/raw/users.csv')
    log.info(f'Get data users done: Total {len(df_users)} rows')

    df_events = pd.read_csv('data/raw/events.csv')
    log.info(f'Get data events done: Total {len(df_events)} rows')

    return df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users

# Transform

In [4]:
def transform(df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users):
    log.info("Transform started!")

    # df_d_center cleaned

    # Transform data df_i_items
    log.info("Transform data df_i_items")
    df_i_items['created_at'] = pd.to_datetime(df_i_items['created_at'], format='ISO8601')
    df_i_items['sold_at'] = pd.to_datetime(df_i_items['sold_at'], format='ISO8601')
    df_i_items['product_name'] = df_i_items['product_name'].fillna('Unknown Product')
    df_i_items['product_brand'] = df_i_items['product_brand'].fillna('Unknown Brand')
    df_i_items["time_to_sell"] = df_i_items["sold_at"] - df_i_items["created_at"]
    log.info("Transform data df_i_items done!")


    # Transform data df_o_items
    log.info("Transform data df_o_items")
    date_cols = ['created_at', 'shipped_at', 'delivered_at', 'returned_at']
    df_o_items[date_cols] = df_o_items[date_cols].apply(pd.to_datetime, format='ISO8601')
    log.info("Transform data df_o_items done!")


    # Transform data df_orders
    log.info("Transform data df_orders")
    date_cols = ['created_at', 'shipped_at', 'delivered_at', 'returned_at']
    df_orders[date_cols] = df_orders[date_cols].apply(pd.to_datetime, format='ISO8601')
    log.info("Transform data df_orders done!")

    # Transform data df_products
    log.info("Transform data df_products")
    df_products['name'] = df_products['name'].fillna('Unknown Product')
    df_products['brand'] = df_products['brand'].fillna('Unknown Brand')
    log.info("Transform data df_products done!")

    # Transform data df_users
    log.info("Transform data df_users")
    df_users['created_at'] = pd.to_datetime(df_users['created_at'], format='ISO8601')
    df_users['city'] = df_users['city'].fillna('Unknown City')
    log.info("Transform data df_users done!")

    # Transform data df_events
    log.info("Transform data df_events")
    df_events['created_at'] = pd.to_datetime(df_events['created_at'], format='ISO8601')
    df_events['city'] = df_events['city'].fillna('Unknown City')
    log.info("Transform data df_events done!")


    log.info("Transform done!")

    return df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users

# Load

In [8]:
def load(df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users):
    log.info("Load started!")

    # Load data to database
    engine = make_engine(LOOKER_ECOMMERCE)
    
    df_d_center.to_sql('distribution_centers', engine, if_exists='replace', index=False)
    df_events.to_sql('events', engine, if_exists='replace', index=False)
    df_i_items.to_sql('inventory_items', engine, if_exists='replace', index=False)
    df_o_items.to_sql('order_items', engine, if_exists='replace', index=False)
    df_orders.to_sql('orders', engine, if_exists='replace', index=False)
    df_products.to_sql('products', engine, if_exists='replace', index=False)
    df_users.to_sql('users', engine, if_exists='replace', index=False)

    log.info("Load done!")

# Main execute

In [9]:
df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users = extract()
df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users = transform(df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users)
load(df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users)

03-11_2026 14:54:24 | INFO | Extract started!
03-11_2026 14:54:24 | INFO | Get data distribution_centers done: Total 10 rows
03-11_2026 14:54:25 | INFO | Get data inventory_items done: Total 490705 rows
03-11_2026 14:54:25 | INFO | Get data order_items done: Total 181759 rows
03-11_2026 14:54:25 | INFO | Get data orders done: Total 125226 rows
03-11_2026 14:54:25 | INFO | Get data products done: Total 29120 rows
03-11_2026 14:54:25 | INFO | Get data users done: Total 100000 rows
03-11_2026 14:54:30 | INFO | Get data events done: Total 2431963 rows
03-11_2026 14:54:30 | INFO | Transform started!
03-11_2026 14:54:30 | INFO | Transform data df_i_items
03-11_2026 14:54:30 | INFO | Transform data df_i_items done!
03-11_2026 14:54:30 | INFO | Transform data df_o_items
03-11_2026 14:54:30 | INFO | Transform data df_o_items done!
03-11_2026 14:54:30 | INFO | Transform data df_orders
03-11_2026 14:54:31 | INFO | Transform data df_orders done!
03-11_2026 14:54:31 | INFO | Transform data df_produ

NameError: name 'LOOKER_ECOMMERCE' is not defined